# 01 Build Static Features

Aim of this notebook: construct the account-window feature table used in the static NHS analysis.

Feature assessment and model selection are performed in later notebooks.


## 0. Configuration


In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np

import nhs_feature_utils as fu


# Data and notebook-00 inputs
AUTH_PATH = "auth.txt.gz"
AUTH_PARQUET = Path("auth_main.parquet")

ACCOUNT_SPLIT_PATH = Path("account_split_static.json")
QUIET_HOURS_PATH = Path("quiet_hours.json")
QUIET_HOURS_META_PATH = Path("quiet_hours_meta.json")


# Feature outputs
FEATURE_DIR = Path("static_features_v2")
FEATURE_DIR.mkdir(exist_ok=True)

BASE_FEATURES_PATH = FEATURE_DIR / "feat_base.parquet"
PERIOD_FEATURES_PATH = FEATURE_DIR / "feat_periodicity.parquet"
FINAL_FEATURES_PATH = FEATURE_DIR / "feat_final.parquet"
PERIOD_PARTS_DIR = FEATURE_DIR / "periodicity_parts"
BUILD_META_PATH = FEATURE_DIR / "feature_build_meta.json"


# Analysis settings
WINDOW_DAYS = 14
MIN_AGG = 20       # minimum events per account-window
MIN_PERIOD = 50    # minimum events for periodicity

BIN_S = 10
MAX_PERIOD_S = 6 * 3600
PERIOD_CHUNK_SIZE = 1000

FORCE_AUTH_REBUILD = False

print("Configuration ready.")


Configuration ready.


## 1. Load authentication events and notebook-00 inputs

The working event stream contains successful `LogOn` events in the four complete 14-day windows. The fixed account split and enterprise-wide quiet hours are loaded from notebook 00.


In [2]:
con, MAX_T, N_WIN = fu.initialise_duckdb(
    auth_parquet=AUTH_PARQUET,
    auth_path=AUTH_PATH,
    window_days=WINDOW_DAYS,
    force_auth_rebuild=FORCE_AUTH_REBUILD,
)

event_summary = con.execute("""
SELECT
    COUNT(*) AS n_logon_events,
    COUNT(DISTINCT u) AS n_source_accounts
FROM e
""").df()


# Account split
if not ACCOUNT_SPLIT_PATH.exists():
    raise FileNotFoundError(f"Missing {ACCOUNT_SPLIT_PATH}. Run notebook 00 first.")

split_obj = json.loads(ACCOUNT_SPLIT_PATH.read_text())
if set(split_obj) != {"fit", "calibration", "test"}:
    raise RuntimeError("Unexpected account-split structure.")

fit_a = set(map(str, split_obj["fit"]))
cal_a = set(map(str, split_obj["calibration"]))
te_a = set(map(str, split_obj["test"]))
eligible_accounts = fit_a | cal_a | te_a

assert fit_a.isdisjoint(cal_a)
assert fit_a.isdisjoint(te_a)
assert cal_a.isdisjoint(te_a)


# Quiet hours
for path in [QUIET_HOURS_PATH, QUIET_HOURS_META_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run notebook 00 first.")

QUIET_HOURS = [int(x) for x in json.loads(QUIET_HOURS_PATH.read_text())]
quiet_meta = json.loads(QUIET_HOURS_META_PATH.read_text())

if quiet_meta.get("scope") != "enterprise_wide":
    raise RuntimeError("Unexpected quiet-hour definition.")

print(event_summary.to_string(index=False))
print(f"Complete windows : {N_WIN}")
print(f"Eligible accounts: {len(eligible_accounts):,}")
print(f"Quiet hours      : {QUIET_HOURS}")


Using existing auth_main.parquet
Success events: 1,038,590,151
Complete 14-day windows: 4
Discarded tail: 172799 seconds
DuckDB views restored: a, e


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 n_logon_events  n_source_accounts
      444450050              54023
Complete windows : 4
Eligible accounts: 29,558
Quiet hours      : [1, 2, 3, 4, 5, 20, 21, 23]


## 2. Construct aggregate features

For each eligible account-window, compute event volume, destination fanout, active days, quiet-hour fraction and inter-arrival-time CV. `ticket_frac` and `logon_miss_frac` are also retained as authentication-record composition features for later assessment.


In [3]:
feat_base = fu.compute_base_features(
    con,
    quiet_hours=QUIET_HOURS,
    base_features_path=BASE_FEATURES_PATH,
    window_days=WINDOW_DAYS,
    n_win=N_WIN,
    min_agg=MIN_AGG,
    force=False,
)

feat_base["u"] = feat_base["u"].astype(str)

if feat_base.duplicated(["u", "win"]).any():
    raise RuntimeError("Duplicate (u, win) rows in base features.")

if set(feat_base["u"]) != eligible_accounts:
    raise RuntimeError("Base-feature account universe does not match notebook 00.")

print(f"Base features: {feat_base.shape}")
print(
    feat_base.groupby("win")
    .agg(rows=("u", "size"))
    .to_string()
)


Loaded base features: (104147, 9) from static_features_v2/feat_base.parquet
Base features: (104147, 9)
      rows
win       
0    26120
1    25801
2    26155
3    26071


## 3. Construct periodicity strength

`period_strength` is calculated for account-windows with at least 50 successful `LogOn` events using 10-second `log1p` count bins, FFT spectral power and periods up to six hours. The Fisher-type score is bounded to `[0, 300]`.


In [4]:
feat_period = fu.compute_periodicity(
    con,
    period_parts_dir=PERIOD_PARTS_DIR,
    period_features_path=PERIOD_FEATURES_PATH,
    n_win=N_WIN,
    window_days=WINDOW_DAYS,
    min_period=MIN_PERIOD,
    chunk_size=PERIOD_CHUNK_SIZE,
    bin_s=BIN_S,
    max_period_s=MAX_PERIOD_S,
    force=False,
)

feat_period["u"] = feat_period["u"].astype(str)

if feat_period.duplicated(["u", "win"]).any():
    raise RuntimeError("Duplicate (u, win) rows in periodicity features.")

finite_period = feat_period["period_strength"].dropna()
if (finite_period < 0).any() or (finite_period > 300).any():
    raise RuntimeError("period_strength must lie in [0, 300].")

print(f"Periodicity features: {feat_period.shape}")
print(f"Coverage: {len(feat_period) / len(feat_base):.2%}")
print(
    "Score range:",
    (float(finite_period.min()), float(finite_period.max())),
)


Loaded periodicity table: (99458, 3) from static_features_v2/feat_periodicity.parquet
Periodicity features: (99458, 3)
Coverage: 95.50%
Score range: (0.0, 300.0)


## 4. Build the final feature table

Merge the aggregate and periodicity features, then add `log_volume = log10(volume)` and `log_fanout = log10(fanout + 1)`.


In [5]:
feat = fu.build_final_features(
    con,
    base_features_path=BASE_FEATURES_PATH,
    period_features_path=PERIOD_FEATURES_PATH,
    final_features_path=FINAL_FEATURES_PATH,
    compatibility_path=None,
    force=True,
)

print("Final feature table:", feat.shape)
print(feat.head().to_string(index=False))


Saved final features: (104147, 12) -> static_features_v2/feat_final.parquet
Final feature table: (104147, 12)
          u  win  volume  fanout  active_days  quiet_frac   iat_cv  ticket_frac  logon_miss_frac  period_strength  log_volume  log_fanout
C5165$@DOM1    0    3881      12           14    0.334708 1.227957     0.095457         0.095457       300.000000    3.588944    1.113943
C1473$@DOM1    0    6139      15           14    0.305261 2.894668     0.107846         0.110183       300.000000    3.788098    1.204120
C4254$@DOM1    0    5582      13           14    0.298818 3.287526     0.097241         0.097241       187.853797    3.746790    1.146128
C2254$@DOM1    0   30064      28           14    0.255122 1.963554     0.077399         0.078903       300.000000    4.478047    1.462398
C1938$@DOM1    0   33148       5           14    0.340232 0.537960     0.503984         0.503984       300.000000    4.520457    0.778151


## 5. Integrity checks and provenance

Check the final account-window table and record hashes of the inputs, helper source and outputs used in this feature build.


In [6]:
EXPECTED_FINAL_COLUMNS = [
    "u",
    "win",
    "volume",
    "fanout",
    "active_days",
    "quiet_frac",
    "iat_cv",
    "ticket_frac",
    "logon_miss_frac",
    "period_strength",
    "log_volume",
    "log_fanout",
]

missing = [c for c in EXPECTED_FINAL_COLUMNS if c not in feat.columns]
if missing:
    raise RuntimeError(f"Final table missing columns: {missing}")

if feat.duplicated(["u", "win"]).any():
    raise RuntimeError("Duplicate (u, win) rows in final feature table.")

if set(feat["u"].astype(str)) != eligible_accounts:
    raise RuntimeError("Final feature-table account universe does not match notebook 00.")

if sorted(feat["win"].astype(int).unique()) != list(range(N_WIN)):
    raise RuntimeError("Unexpected feature-table windows.")

if (feat["volume"] < MIN_AGG).any():
    raise RuntimeError("Final table contains rows below the event threshold.")

finite_period = feat["period_strength"].dropna()
if (finite_period < 0).any() or (finite_period > 300).any():
    raise RuntimeError("period_strength must lie in [0, 300].")

if not np.allclose(
    feat["log_volume"],
    np.log10(feat["volume"].clip(lower=1)),
):
    raise RuntimeError("log_volume transform mismatch.")

if not np.allclose(
    feat["log_fanout"],
    np.log10(feat["fanout"].clip(lower=0) + 1),
):
    raise RuntimeError("log_fanout transform mismatch.")


def file_sha256(path):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def hash_account_set(accounts):
    payload = "\n".join(sorted(map(str, accounts))).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


build_meta = {
    "pipeline_stage": "01_build_static_features",
    "window_days": WINDOW_DAYS,
    "min_agg": MIN_AGG,
    "min_period": MIN_PERIOD,
    "periodicity": {
        "bin_seconds": BIN_S,
        "max_period_seconds": MAX_PERIOD_S,
        "score_range": [0.0, 300.0],
    },
    "quiet_hours": QUIET_HOURS,
    "n_complete_windows": N_WIN,
    "n_final_rows": int(len(feat)),
    "n_final_accounts": int(feat["u"].nunique()),
    "eligible_account_sha256": hash_account_set(eligible_accounts),
    "feature_helper_sha256": file_sha256(Path(fu.__file__)),
    "input_hashes": {
        "auth_main_parquet": file_sha256(AUTH_PARQUET),
        "account_split_static_json": file_sha256(ACCOUNT_SPLIT_PATH),
        "quiet_hours_json": file_sha256(QUIET_HOURS_PATH),
    },
    "output_hashes": {
        "feat_base": file_sha256(BASE_FEATURES_PATH),
        "feat_periodicity": file_sha256(PERIOD_FEATURES_PATH),
        "feat_final": file_sha256(FINAL_FEATURES_PATH),
    },
}

BUILD_META_PATH.write_text(json.dumps(build_meta, indent=2))

print("Integrity checks passed.")
print(f"Rows / accounts: {len(feat):,} / {feat['u'].nunique():,}")
print(f"Periodicity available: {feat['period_strength'].notna().mean():.2%}")
print(f"Saved: {FINAL_FEATURES_PATH}")


Integrity checks passed.
Rows / accounts: 104,147 / 29,558
Periodicity available: 95.50%
Saved: static_features_v2/feat_final.parquet
